In [1]:
from google.colab import userdata
userdata.get('WEATHER_API_KEY')
from google.colab import userdata
userdata.get('Groq_api')
Groq_api="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"
WEATHER_API_KEY="37954f583353278cf114866bc8d2a27c"


In [2]:
!pip install -qU langchain langchain-groq langchain-community langchain-core requests duckduckgo-search ddgs langsmith langgraph

In [3]:
!pip show langchain langchain-groq langchain-community langchain-core requests duckduckgo-search ddgs langsmith langgraph

Name: langchain
Version: 1.3.11
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-groq
Version: 1.1.3
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: 
---
Name: langchain-core
Ve

#import core library

In [4]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests

#set up inbuilt

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

/tmp/ipykernel_2493/2272844035.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


#live tool

In [6]:
@tool
def get_weather_data(city: str) -> str:
  """
  this tool fetches the live weather data of a given city
  """
  url = f'https://api.weatherstack.com/current?access_key={WEATHER_API_KEY}&query={city}'

  response = requests.get(url)

  return response.json()

#grok model initialization

In [7]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=Groq_api,
    temperature=0,
    model_kwargs={"tool_choice": "auto"}
)

#Import agent Components

In [8]:
from langchain.agents import create_agent
from langsmith import Client

In [11]:
!pip install -U langchain langsmith langchain-core

In [12]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt("hwchase17/react")
prompt_template_string = prompt.template

ValueError: Pulling a public prompt by owner/name is disabled by default because prompts may contain untrusted serialized LangChain objects. If you trust this prompt, set `dangerously_pull_public_prompt=True` to acknowledge the risk.

In [ ]:
prompt.template

In [ ]:
agent = create_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    system_prompt=prompt_template_string
)

In [ ]:
response = agent.invoke(
    {"messages": ("user", "find the capital of rajastan,then find it's weather condition")}
)
print(response)

#Deploy with Gradio

In [ ]:
def get_response(query):
  response = agent.invoke(
    {"messages": ("user", query)}
  )
  return response['messages'][-1].content

In [ ]:
get_response("find the capital of rajasthan, then find its weather condition")

In [ ]:
import gradio as gr

iface = gr.Interface(
    fn=get_response,
    inputs = gr.Textbox(lines=2, placeholder="Enter text and press enter"),
    outputs = gr.Textbox(
        label = "Response",
        lines = 10
    ),
    title = "AI Agent with web access",
    description = "this is ai agent working with web"
)

In [ ]:
iface.launch(share=True)